# Boiler Digital Twin — End-to-End Notebook

All pipeline steps in one place, with plots at each stage.

| Step | Content |
|------|---------|
| 1 | DCS Parsing & Causal Graph |
| 2 | Data Exploration |
| 3 | Dataset Building |
| 4 | Model Architecture |
| 5 | Phase A — Train Controllers |
| 6 | Phase B — Train Plant LSTM |
| 7 | Closed-Loop Rollout |
| 8 | Level 1 & 2 Validation |
| 9 | Full Evaluation Plots |

In [ ]:
import ast, json, os, sys, time
from pathlib import Path
from typing import Dict, List, Optional, Tuple

import numpy as np
import pandas as pd
import matplotlib
import matplotlib.pyplot as plt
import matplotlib.figure
from matplotlib.figure import Figure as MplFigure
from scipy.stats import probplot, ks_2samp
from sklearn.preprocessing import StandardScaler
import joblib
import torch
import torch.nn as nn
from torch.utils.data import DataLoader, TensorDataset

# ── Paths ─────────────────────────────────────────────────────────────
ROOT        = Path('C:/Users/ahmma/Desktop/farah/hai-digital-twin')
BOILER_DIR  = Path('C:/Users/ahmma/Desktop/farah/boiler')
DATA_DIR    = ROOT / 'data' / 'processed'
OUTPUT_DIR  = ROOT / 'outputs' / 'boiler_twin'
(OUTPUT_DIR / 'models').mkdir(parents=True, exist_ok=True)
(OUTPUT_DIR / 'plots').mkdir(parents=True, exist_ok=True)

# ── Device ────────────────────────────────────────────────────────────
DEVICE = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f'Device: {DEVICE}')

# ── Scenario labels ───────────────────────────────────────────────────
SCENARIO_NAMES  = {0:'Normal', 1:'AP no-comb', 2:'AP with-comb', 3:'AE no-comb', 4:'AE with-comb'}
SCENARIO_COLORS = {0:'#2ca02c', 1:'#ff7f0e', 2:'#d62728', 3:'#9467bd', 4:'#8c564b'}
print('Setup complete.')

---
## Step 1 — DCS Parsing & Causal Graph

Parse the boiler DCS JSON files to extract:
- Control loop definitions (SP / PV / CV / CV_fb)
- Attack target nodes (yellow nodes in each DCS diagram)
- Physical causal adjacency from `phy_boiler.json`

In [ ]:
# ── DCS file paths ────────────────────────────────────────────────────
DCS_FILES = {
    'PC':    BOILER_DIR / 'dcs_1001h.json',
    'LC':    BOILER_DIR / 'dcs_1002h.json',
    'FC_TC': BOILER_DIR / 'dcs_1003h.json',
    'HTR':   BOILER_DIR / 'dcs_1004h.json',
    'EXT1':  BOILER_DIR / 'dcs_1010h.json',
    'EXT2':  BOILER_DIR / 'dcs_1011h.json',
    'CC':    BOILER_DIR / 'dcs_1020h.json',
}
PHY_FILE = BOILER_DIR / 'phy_boiler.json'

# ── Loop definitions ──────────────────────────────────────────────────
LOOPS_DEF = {
    'PC': {'sp': 'x1001_05_SETPOINT_OUT', 'pv': 'P1_PIT01',  'cv': 'P1_PCV01D', 'cv_fb': 'P1_PCV01Z'},
    'LC': {'sp': 'x1002_07_SETPOINT_OUT', 'pv': 'P1_LIT01',  'cv': 'P1_LCV01D', 'cv_fb': 'P1_LCV01Z'},
    'FC': {'sp': 'x1002_08_SETPOINT_OUT', 'pv': 'P1_FT03Z',  'cv': 'P1_FCV03D', 'cv_fb': 'P1_FCV03Z'},
    'TC': {'sp': 'x1003_18_SETPOINT_OUT', 'pv': 'P1_TIT01',  'cv': 'P1_FCV01D', 'cv_fb': 'P1_FCV01Z'},
    'CC': {'sp': 'P1_PP04SP',             'pv': 'P1_TIT03',  'cv': 'P1_PP04',   'cv_fb': None},
}
PV_COLS        = [d['pv'] for d in LOOPS_DEF.values()]
CV_COLS        = [d['cv'] for d in LOOPS_DEF.values()]
PLANT_AUX_COLS = [
    'P1_PP01AD','P1_PP01BD','P1_PP02D',
    'P1_FT01','P1_FT01Z','P1_FT02','P1_FT02Z','P1_FT03',
    'P1_PIT02','P1_TIT02',
    'x1001_15_ASSIGN_OUT','x1003_10_SETPOINT_OUT','x1003_24_SUM_OUT',
    'GATEOPEN',
]
PLANT_IN_COLS  = CV_COLS + PLANT_AUX_COLS
PLANT_OUT_COLS = PV_COLS

print('PV_COLS :', PV_COLS)
print('CV_COLS :', CV_COLS)
print(f'Plant inputs: {len(PLANT_IN_COLS)}  (CVs={len(CV_COLS)}, aux={len(PLANT_AUX_COLS)})')
print(f'Plant outputs: {len(PLANT_OUT_COLS)}')

In [ ]:
# ── Loader ────────────────────────────────────────────────────────────
def _load_dcs(path):
    with open(path, 'r') as f:
        text = f.read()
    try:
        return json.loads(text)
    except json.JSONDecodeError:
        return ast.literal_eval(text)

# ── Attack targets ────────────────────────────────────────────────────
def build_attack_targets():
    targets = {}
    for ln, path in DCS_FILES.items():
        if path.exists():
            data = _load_dcs(path)
            targets[ln] = [n['id'] for n in data['nodes'] if n.get('fillcolor') == 'yellow']
        else:
            targets[ln] = []
    return targets

# ── Causal adjacency ──────────────────────────────────────────────────
NODE_TO_CSV = {
    'PCV01':'P1_PCV01D','PCV02':'P1_PCV02D','LCV01':'P1_LCV01D',
    'FCV03':'P1_FCV03D','FCV01':'P1_FCV01D','FCV02':'P1_FCV02D',
    'PP01':'P1_PP01AD','PP02':'P1_PP02D','PP04':'P1_PP04',
    'PIT01':'P1_PIT01','PIT02':'P1_PIT02','LIT01':'P1_LIT01',
    'FT01':'P1_FT01','FT03':'P1_FT03Z','TIT01':'P1_TIT01',
    'TIT02':'P1_TIT02','TIT03':'P1_TIT03',
}

def build_causal_adjacency():
    data  = _load_dcs(PHY_FILE)
    edges = [(lk['source'], lk['target']) for lk in data['links']]
    adj   = {pv: [] for pv in PV_COLS}
    for src, tgt in edges:
        s = NODE_TO_CSV.get(src)
        t = NODE_TO_CSV.get(tgt)
        if s in CV_COLS and t in PV_COLS and s not in adj[t]:
            adj[t].append(s)
    return adj

# ── Run ───────────────────────────────────────────────────────────────
print('=' * 55)
print('LOOPS DEFINITION')
print('=' * 55)
for ln, d in LOOPS_DEF.items():
    print(f'  {ln}: SP={d["sp"][:22]}  PV={d["pv"]}  CV={d["cv"]}  fb={d["cv_fb"]}')

print('\n' + '=' * 55)
print('ATTACK TARGETS')
print('=' * 55)
ATTACK_TARGETS = build_attack_targets()
for ln, nodes in ATTACK_TARGETS.items():
    print(f'  {ln}: {nodes}')

print('\n' + '=' * 55)
print('CAUSAL ADJACENCY  (PV <- CVs)')
print('=' * 55)
CAUSAL_ADJ = build_causal_adjacency()
for pv, cvs in CAUSAL_ADJ.items():
    print(f'  {pv} <- {cvs if cvs else "(no direct CV edge)"}')

In [ ]:
# Plot: Causal adjacency matrix heatmap
mask = np.zeros((len(PV_COLS), len(CV_COLS)))
for i, pv in enumerate(PV_COLS):
    for j, cv in enumerate(CV_COLS):
        if cv in CAUSAL_ADJ.get(pv, []):
            mask[i, j] = 1.0

fig, ax = plt.subplots(figsize=(7, 4))
im = ax.imshow(mask, cmap='Blues', aspect='auto', vmin=0, vmax=1)
ax.set_xticks(range(len(CV_COLS)))
ax.set_xticklabels([c[:12] for c in CV_COLS], rotation=45, ha='right', fontsize=8)
ax.set_yticks(range(len(PV_COLS)))
ax.set_yticklabels(PV_COLS, fontsize=9)
for i in range(len(PV_COLS)):
    for j in range(len(CV_COLS)):
        ax.text(j, i, 'X' if mask[i,j] else '', ha='center', va='center', fontsize=12, fontweight='bold')
ax.set_title('Physical Causal Adjacency  (CV -> PV)', fontsize=12)
ax.set_xlabel('CV (cause)'); ax.set_ylabel('PV (effect)')
plt.colorbar(im, ax=ax, label='Edge exists')
fig.tight_layout(); plt.show()
fig.savefig(OUTPUT_DIR / 'plots' / 'causal_heatmap.png', dpi=150, bbox_inches='tight')
print('Saved causal_heatmap.png')

---
## Step 2 — Data Exploration

Load the processed CSV files and visualise the key PV signals and attack label distributions.

In [ ]:
META_COLS = {
    'timestamp','label','attack_id','scenario','attack_type','combination',
    'target_controller','target_points','duration_sec','episode_id',
    'source_file','scenario_label',
}

def _load_csv(path):
    df = pd.read_csv(path, low_memory=False)
    df['timestamp'] = pd.to_datetime(df['timestamp'])
    return df.sort_values('timestamp').reset_index(drop=True)

def _sensor_cols(df):
    return [c for c in df.columns if c not in META_COLS and pd.api.types.is_numeric_dtype(df[c])]

# Load all files
files = {}
for name in ['train1','train2','train3','train4','test1','test2']:
    p = DATA_DIR / f'{name}.csv'
    if p.exists():
        files[name] = _load_csv(p)
        print(f'{name}: {len(files[name]):>7,} rows  '
              f'attack={int((files[name]["label"]>0).sum()):>6,} rows  '
              f'({100*(files[name]["label"]>0).mean():.1f}%)')
    else:
        print(f'{name}: NOT FOUND')

SENSOR_COLS = _sensor_cols(files['train1'])
print(f'\nSensor features: {len(SENSOR_COLS)}')

In [ ]:
# Plot: PV signals across all CSV files
pv_available = [p for p in PV_COLS if p in files['train1'].columns]
n_pv = len(pv_available)
plot_files = ['train1','train2','train3','train4']

fig, axes = plt.subplots(n_pv, len(plot_files), figsize=(16, 3*n_pv), sharex='col', sharey='row', squeeze=False)
for col, fname in enumerate(plot_files):
    if fname not in files: continue
    df = files[fname]
    t  = np.arange(len(df)) / 3600  # hours
    for row, pv in enumerate(pv_available):
        ax = axes[row][col]
        ax.plot(t, df[pv].values, lw=0.5, color='steelblue', alpha=0.8)
        if 'label' in df.columns:
            atk = df['label'].values > 0
            if atk.any():
                ax.fill_between(t, df[pv].min(), df[pv].max(), where=atk,
                                alpha=0.25, color='red', label='Attack' if row==0 else '_')
        if col == 0: ax.set_ylabel(pv[:16], fontsize=8)
        if row == 0: ax.set_title(fname, fontsize=10)
        ax.grid(True, alpha=0.15)
    axes[-1][col].set_xlabel('Time (h)', fontsize=8)

fig.suptitle('PV Signals Across All Files (red = attack region)', fontsize=12)
fig.tight_layout(); plt.show()
fig.savefig(OUTPUT_DIR / 'plots' / 'pv_overview.png', dpi=120, bbox_inches='tight')

In [ ]:
# Plot: Attack label distribution per file
fig, axes = plt.subplots(1, 2, figsize=(12, 4))

# Bar chart: attack vs normal rows per file
names, normal_counts, attack_counts = [], [], []
for fname, df in files.items():
    names.append(fname)
    n_atk = int((df['label'] > 0).sum()) if 'label' in df.columns else 0
    normal_counts.append(len(df) - n_atk)
    attack_counts.append(n_atk)

x = np.arange(len(names))
axes[0].bar(x, normal_counts, label='Normal', color='#2ca02c', alpha=0.8)
axes[0].bar(x, attack_counts, bottom=normal_counts, label='Attack', color='#d62728', alpha=0.8)
axes[0].set_xticks(x); axes[0].set_xticklabels(names, rotation=30)
axes[0].set_ylabel('Rows'); axes[0].set_title('Normal vs Attack Rows per File')
axes[0].legend(); axes[0].grid(True, alpha=0.2, axis='y')

# Scenario breakdown in test files
sc_counts = {sc: 0 for sc in range(5)}
for fname in ['test1','test2']:
    if fname not in files: continue
    df = files[fname]
    if 'attack_type' not in df.columns: continue
    for _, row in df[df['label']>0].drop_duplicates('attack_id').iterrows():
        at   = row.get('attack_type', 'normal')
        comb = row.get('combination', 'no')
        sc   = 0
        if at == 'AP': sc = 1 if str(comb).lower()=='no' else 2
        elif at == 'AE': sc = 3 if str(comb).lower()=='no' else 4
        sc_counts[sc] += 1

labels = [SCENARIO_NAMES[sc] for sc in range(5)]
colors = [SCENARIO_COLORS[sc] for sc in range(5)]
values = [sc_counts[sc] for sc in range(5)]
axes[1].bar(range(5), values, color=colors, alpha=0.85, edgecolor='white')
axes[1].set_xticks(range(5)); axes[1].set_xticklabels(labels, rotation=25, fontsize=9)
axes[1].set_ylabel('Attack instances'); axes[1].set_title('Attack Scenario Distribution')
axes[1].grid(True, alpha=0.2, axis='y')

fig.tight_layout(); plt.show()
fig.savefig(OUTPUT_DIR / 'plots' / 'attack_distribution.png', dpi=120, bbox_inches='tight')

---
## Step 3 — Build Datasets

- Scaler fitted on **normal data from train1-3 only**
- train4 completely held out for testing
- 80% of attack segments → train, 20% → test
- Scenario labels: 0=normal, 1=AP_no, 2=AP_with, 3=AE_no, 4=AE_with
- Sequence length: 300 steps (5 min at 1 Hz)

In [ ]:
SEQ_LEN            = 300
STRIDE             = 10
TRAIN_RATIO        = 0.80
ATTACK_SPLIT_RATIO = 0.80
BEFORE_ATTACK_SEC  = 300
AFTER_ATTACK_SEC   = 180

def get_scenario_label(attack_type, combination='no'):
    if pd.isna(attack_type) or attack_type == 'normal': return 0
    if attack_type == 'AP': return 1 if str(combination).strip().lower() == 'no' else 2
    if attack_type == 'AE': return 3 if str(combination).strip().lower() == 'no' else 4
    return 0

def _extract_attack_segments(df, before_sec=BEFORE_ATTACK_SEC, after_sec=AFTER_ATTACK_SEC):
    if 'label' not in df.columns or not (df['label'] > 0).any(): return []
    mask   = (df['label'] > 0).values
    blocks = []
    i = 0
    while i < len(mask):
        if mask[i]:
            j = i
            while j < len(mask) and mask[j]: j += 1
            blocks.append((i, j - 1)); i = j
        else: i += 1
    segments = []
    for start, end in blocks:
        seg = df.iloc[max(0,start-before_sec):min(len(df)-1,end+after_sec)+1].copy().reset_index(drop=True)
        atk = seg[seg['label'] > 0]
        if len(atk):
            seg['scenario_label'] = get_scenario_label(atk.iloc[0]['attack_type'], atk.iloc[0].get('combination','no'))
        segments.append(seg)
    return segments

def _make_ctrl_sequences(df, loop_name, scaler, all_cols, seq_len=SEQ_LEN, stride=STRIDE):
    d      = LOOPS_DEF[loop_name]
    cols   = [d['sp'], d['pv']] + ([d['cv_fb']] if d['cv_fb'] else []) + [d['cv']]
    cols   = [c for c in cols if c in df.columns and c in all_cols]
    cidx   = {c: all_cols.index(c) for c in cols}
    raw    = df[cols].values.astype(np.float32)
    full   = np.zeros((len(df), len(all_cols)), np.float32)
    for i, c in enumerate(cols): full[:, cidx[c]] = raw[:, i]
    scaled = scaler.transform(full)
    sc_col = np.column_stack([scaled[:, cidx[c]] for c in cols])
    n_in   = len(cols) - 1
    n_seq  = (len(df) - seq_len) // stride
    inp    = np.zeros((n_seq, seq_len, n_in), np.float32)
    tgt    = np.zeros((n_seq, seq_len), np.float32)
    scn    = np.zeros(n_seq, np.int32)
    has_sc = 'scenario_label' in df.columns
    for i in range(n_seq):
        s = i * stride
        inp[i] = sc_col[s:s+seq_len, :-1]
        tgt[i] = sc_col[s:s+seq_len, -1]
        if has_sc:
            w = df['scenario_label'].iloc[s:s+seq_len]
            scn[i] = int(w[w>0].mode()[0]) if (w>0).any() else 0
    return inp, tgt, scn

def _make_plant_sequences(df, scaler, all_cols, seq_len=SEQ_LEN, stride=STRIDE):
    in_c  = [c for c in PLANT_IN_COLS  if c in df.columns and c in all_cols]
    out_c = [c for c in PLANT_OUT_COLS if c in df.columns and c in all_cols]
    pcols = in_c + out_c
    cidx  = {c: all_cols.index(c) for c in pcols}
    raw   = df[pcols].values.astype(np.float32)
    full  = np.zeros((len(df), len(all_cols)), np.float32)
    for i, c in enumerate(pcols): full[:, cidx[c]] = raw[:, i]
    scaled = scaler.transform(full)
    sc_col = np.column_stack([scaled[:, cidx[c]] for c in pcols])
    ni, no = len(in_c), len(out_c)
    n_seq  = (len(df) - seq_len - 1) // stride
    cv_s   = np.zeros((n_seq, seq_len, ni), np.float32)
    pi_s   = np.zeros((n_seq, no), np.float32)
    pte_s  = np.zeros((n_seq, seq_len, no), np.float32)
    ptr_s  = np.zeros((n_seq, seq_len, no), np.float32)
    scn    = np.zeros(n_seq, np.int32)
    has_sc = 'scenario_label' in df.columns
    for i in range(n_seq):
        s = i * stride
        cv_s[i]  = sc_col[s:s+seq_len, :ni]
        pi_s[i]  = sc_col[s, ni:]
        pte_s[i] = sc_col[s:s+seq_len, ni:]
        ptr_s[i] = sc_col[s+1:s+seq_len+1, ni:]
        if has_sc:
            w = df['scenario_label'].iloc[s:s+seq_len]
            scn[i] = int(w[w>0].mode()[0]) if (w>0).any() else 0
    return cv_s, pi_s, pte_s, ptr_s, scn
print('Dataset helper functions defined.')

In [ ]:
print('Building datasets...')
train_eps, val_eps = [], []
for n in range(1, 4):
    df = files[f'train{n}'].copy()
    df['scenario_label'] = 0
    cut = int(len(df) * TRAIN_RATIO)
    tr, va = df.iloc[:cut].copy(), df.iloc[cut:].copy()
    tr['episode_id'] = va['episode_id'] = f'train{n}'
    train_eps.append(tr); val_eps.append(va)
    print(f'  train{n}: {len(df):,} rows -> train {len(tr):,} / val {len(va):,}')

train4 = files['train4'].copy()
train4['scenario_label'] = 0
train4['episode_id'] = 'train4_TEST'
print(f'  train4 (held out): {len(train4):,} rows')

all_attack_segs = []
for n in range(1, 3):
    if f'test{n}' in files:
        segs = _extract_attack_segments(files[f'test{n}'])
        all_attack_segs.extend(segs)
        print(f'  test{n}: {len(segs)} attack segments')

np.random.seed(42)
idxs = np.random.permutation(len(all_attack_segs))
cut  = int(len(idxs) * ATTACK_SPLIT_RATIO)
atk_train = [all_attack_segs[i] for i in idxs[:cut]]
atk_test  = [all_attack_segs[i] for i in idxs[cut:]]
print(f'  Attack segments: {len(atk_train)} train / {len(atk_test)} test')

# Fit scaler on normal data only
normal_data = pd.concat(
    [ep[ep['label']==0] if 'label' in ep.columns else ep for ep in train_eps],
    ignore_index=True)
scaler = StandardScaler()
scaler.fit(normal_data[SENSOR_COLS].values.astype(np.float32))
joblib.dump(scaler, OUTPUT_DIR / 'scaler.pkl')
np.save(OUTPUT_DIR / 'scaler_cols.npy', np.array(SENSOR_COLS))
print(f'\nScaler fitted on {len(normal_data):,} normal rows, {len(SENSOR_COLS)} features.')

train_all = train_eps + atk_train
test_all  = [train4] + atk_test

# Controller datasets
ctrl_data = {}
for ln in LOOPS_DEF:
    try:
        tr_i,tr_t,tr_s = _make_ctrl_sequences(pd.concat(train_all, ignore_index=True), ln, scaler, SENSOR_COLS)
        va_i,va_t,va_s = _make_ctrl_sequences(pd.concat(val_eps, ignore_index=True),   ln, scaler, SENSOR_COLS)
        ctrl_data[ln] = {'train':(tr_i,tr_t,tr_s), 'val':(va_i,va_t,va_s)}
        print(f'  {ln}: train={len(tr_i):,}  val={len(va_i):,}  input_dim={tr_i.shape[-1]}')
    except Exception as e:
        print(f'  {ln}: WARN - {e}')

# Plant datasets
cv_tr,pi_tr,pte_tr,ptr_tr,sc_tr = _make_plant_sequences(pd.concat(train_all,ignore_index=True), scaler, SENSOR_COLS)
cv_va,pi_va,pte_va,ptr_va,sc_va = _make_plant_sequences(pd.concat(val_eps,  ignore_index=True), scaler, SENSOR_COLS)
N_PLANT_IN  = cv_tr.shape[-1]
N_PLANT_OUT = ptr_tr.shape[-1]
print(f'\nPlant: n_in={N_PLANT_IN}  n_out={N_PLANT_OUT}  train={len(cv_tr):,}  val={len(cv_va):,}')

In [ ]:
# Plot: Sample controller sequences per loop
loops_with_data = [ln for ln in LOOPS_DEF if ln in ctrl_data]
n_loops = len(loops_with_data)
fig, axes = plt.subplots(n_loops, 2, figsize=(14, 3*n_loops), squeeze=False)

for row, ln in enumerate(loops_with_data):
    inp, tgt, scn = ctrl_data[ln]['train']
    # pick one normal and one attack sequence
    norm_idx = np.where(scn == 0)[0]
    atk_idx  = np.where(scn > 0)[0]
    idx_n = norm_idx[0] if len(norm_idx) else 0
    idx_a = atk_idx[0]  if len(atk_idx)  else 0
    t = np.arange(SEQ_LEN)

    for col, (idx, label, color) in enumerate([(idx_n,'Normal','steelblue'),(idx_a,'Attack','crimson')]):
        ax = axes[row][col]
        for i in range(inp.shape[-1]):
            ax.plot(t, inp[idx,:,i], lw=0.9, alpha=0.8, label=['SP','PV','CV_fb'][i] if i<3 else f'in{i}')
        ax.plot(t, tgt[idx], lw=1.2, linestyle='--', color='black', label='CV target')
        ax.set_title(f'{ln} — {label} (sc={scn[idx]})', fontsize=9)
        if col == 0: ax.set_ylabel('Scaled value', fontsize=8)
        ax.legend(fontsize=6, ncol=2); ax.grid(True, alpha=0.2)
axes[-1][0].set_xlabel('Timestep'); axes[-1][1].set_xlabel('Timestep')
fig.suptitle('Sample Controller Sequences (scaled)', fontsize=12)
fig.tight_layout(); plt.show()
fig.savefig(OUTPUT_DIR/'plots'/'sample_sequences.png', dpi=120, bbox_inches='tight')

---
## Step 4 — Model Architecture

**ControllerLSTM** — one per loop, predicts CV from [SP, PV, (CV_fb)] + scenario embedding  
**PlantLSTM** — single MIMO model, predicts all 5 PVs from CVs + aux signals  
**CausalLoss** — penalises spurious correlations that violate the physical causal graph

In [ ]:
class ControllerLSTM(nn.Module):
    def __init__(self, n_ctrl_in, hidden_dim=128, num_layers=2, dropout=0.1, n_scenarios=5, emb_dim=16):
        super().__init__()
        self.scenario_emb = nn.Embedding(n_scenarios, emb_dim)
        self.input_proj   = nn.Linear(n_ctrl_in + emb_dim, hidden_dim)
        self.lstm = nn.LSTM(hidden_dim, hidden_dim, num_layers, batch_first=True,
                            dropout=dropout if num_layers>1 else 0.0)
        self.fc   = nn.Linear(hidden_dim, 1)

    def _proj(self, x, scenario):
        emb = self.scenario_emb(scenario).unsqueeze(1).expand(-1, x.size(1), -1)
        return self.input_proj(torch.cat([x, emb], dim=-1))

    def forward(self, inputs, scenario, h=None):
        x_proj, _ = self.lstm(self._proj(inputs, scenario), h)
        cv_pred    = self.fc(x_proj).squeeze(-1)
        _, h_new   = self.lstm(self._proj(inputs, scenario), h)
        return cv_pred, h_new

    @torch.no_grad()
    def step(self, x_t, scenario, h=None):
        x_t  = x_t.unsqueeze(1)
        emb  = self.scenario_emb(scenario).unsqueeze(1)
        x_in = self.input_proj(torch.cat([x_t, emb], dim=-1))
        out, h_new = self.lstm(x_in, h)
        return self.fc(out[:,0,:]).squeeze(-1), h_new

print('ControllerLSTM defined.')

In [ ]:
class PlantLSTM(nn.Module):
    def __init__(self, n_in, n_out, hidden_dim=256, num_layers=2, dropout=0.1, n_scenarios=5, emb_dim=32):
        super().__init__()
        self.n_in = n_in; self.n_out = n_out
        self.scenario_emb = nn.Embedding(n_scenarios, emb_dim)
        self.lstm = nn.LSTM(n_in+n_out+emb_dim, hidden_dim, num_layers, batch_first=True,
                            dropout=dropout if num_layers>1 else 0.0)
        self.fc   = nn.Sequential(nn.Linear(hidden_dim, hidden_dim//2), nn.ReLU(),
                                   nn.Linear(hidden_dim//2, n_out))

    def _fast(self, x_cv, pv_init, scenario, teacher_pv):
        emb    = self.scenario_emb(scenario).unsqueeze(1).expand(-1, x_cv.size(1), -1)
        pv_in  = torch.cat([pv_init.unsqueeze(1), teacher_pv[:,:-1,:]], dim=1)
        out, _ = self.lstm(torch.cat([x_cv, pv_in, emb], dim=-1))
        return self.fc(out)

    def forward(self, x_cv, pv_init, scenario, teacher_pv=None, ss_ratio=0.0):
        if ss_ratio == 0.0 and teacher_pv is not None:
            return self._fast(x_cv, pv_init, teacher_pv=teacher_pv, scenario=scenario)
        B, T, _ = x_cv.shape
        emb  = self.scenario_emb(scenario)
        pv   = pv_init; h = None; out_list = []
        for t in range(T):
            x_t = torch.cat([x_cv[:,t,:], pv, emb], dim=-1).unsqueeze(1)
            o, h = self.lstm(x_t, h)
            pv_pred = self.fc(o[:,0,:])
            out_list.append(pv_pred)
            if t < T-1:
                if teacher_pv is not None and ss_ratio < 1.0:
                    use = (torch.rand(B, device=x_cv.device) < ss_ratio).float().unsqueeze(-1)
                    pv  = use * pv_pred.detach() + (1-use) * teacher_pv[:,t,:]
                else:
                    pv  = pv_pred.detach()
        return torch.stack(out_list, dim=1)

    @torch.no_grad()
    def step(self, x_cv_t, pv_current, scenario, h=None):
        emb   = self.scenario_emb(scenario)
        x_t   = torch.cat([x_cv_t, pv_current, emb], dim=-1).unsqueeze(1)
        out, h_new = self.lstm(x_t, h)
        return self.fc(out[:,0,:]), h_new

print('PlantLSTM defined.')

In [ ]:
def build_causal_mask(pv_cols, cv_cols, adj):
    mask = torch.zeros(len(pv_cols), len(cv_cols))
    for i, pv in enumerate(pv_cols):
        for j, cv in enumerate(cv_cols):
            if cv in adj.get(pv, []):
                mask[i, j] = 1.0
    return mask

class CausalLoss(nn.Module):
    def __init__(self, pv_cols=PV_COLS, cv_cols=CV_COLS, adj=None, lambda_causal=0.1):
        super().__init__()
        self.lambda_causal = lambda_causal
        if adj is None: adj = CAUSAL_ADJ
        mask = build_causal_mask(pv_cols, cv_cols, adj)
        self.register_buffer('non_causal_mask', 1.0 - mask)

    def forward(self, pv_pred, cv_seqs):
        cv  = cv_seqs[:, :, :len(CV_COLS)]
        def _norm(x):
            mu  = x.mean(dim=1, keepdim=True)
            std = x.std(dim=1, keepdim=True).clamp(min=1e-6)
            return (x - mu) / std
        corr    = torch.einsum('btp,btc->bpc', _norm(pv_pred), _norm(cv)) / pv_pred.size(1)
        mask    = self.non_causal_mask.to(corr.device)
        penalty = (corr.abs() * mask.unsqueeze(0)).sum() / (mask.sum() * corr.size(0) + 1e-8)
        return self.lambda_causal * penalty

def combined_loss(pv_pred, pv_target, cv_seqs, causal_fn):
    mse    = nn.functional.mse_loss(pv_pred, pv_target)
    causal = causal_fn(pv_pred, cv_seqs)
    return mse + causal, mse, causal

print('CausalLoss defined.')

In [ ]:
# Model parameter summary + causal mask heatmap
N_SCENARIOS = 5; EMB_DIM = 16

print('Controller LSTMs:')
ctrl_models = {}
for ln, d in LOOPS_DEF.items():
    n_in = 3 if d['cv_fb'] else 2
    m    = ControllerLSTM(n_in, hidden_dim=128, n_scenarios=N_SCENARIOS, emb_dim=EMB_DIM)
    n_p  = sum(p.numel() for p in m.parameters())
    print(f'  {ln}: n_in={n_in}  params={n_p:,}')
    ctrl_models[ln] = m

plant_model = PlantLSTM(N_PLANT_IN, N_PLANT_OUT, hidden_dim=256, n_scenarios=N_SCENARIOS, emb_dim=EMB_DIM*2)
n_plant = sum(p.numel() for p in plant_model.parameters())
print(f'\nPlant LSTM: n_in={N_PLANT_IN}  n_out={N_PLANT_OUT}  params={n_plant:,}')

# Causal mask heatmap
mask_np = build_causal_mask(PV_COLS, CV_COLS, CAUSAL_ADJ).numpy()
fig, ax  = plt.subplots(figsize=(6,3.5))
im = ax.imshow(mask_np, cmap='Greens', aspect='auto', vmin=0, vmax=1)
ax.set_xticks(range(len(CV_COLS)))
ax.set_xticklabels([c[:12] for c in CV_COLS], rotation=45, ha='right', fontsize=8)
ax.set_yticks(range(len(PV_COLS)))
ax.set_yticklabels(PV_COLS, fontsize=9)
for i in range(len(PV_COLS)):
    for j in range(len(CV_COLS)):
        txt = 'causal' if mask_np[i,j] else 'penalised'
        ax.text(j, i, txt, ha='center', va='center', fontsize=6,
                color='white' if mask_np[i,j] else 'dimgray')
ax.set_title('CausalLoss Mask — green=allowed, white=penalised', fontsize=10)
fig.tight_layout(); plt.show()
fig.savefig(OUTPUT_DIR/'plots'/'causal_mask.png', dpi=150, bbox_inches='tight')

---
## Step 5 — Phase A: Train Controllers

One `ControllerLSTM` per loop. Inputs: `[SP, PV, CV_fb]` + scenario embedding → output: CV.

In [ ]:
CTRL_LR       = 1e-3
CTRL_EPOCHS   = 100
CTRL_BATCH    = 256
CTRL_PATIENCE = 15

ctrl_losses = {}   # {ln: (train_losses, val_losses)}

for ln, model in ctrl_models.items():
    if ln not in ctrl_data:
        print(f'[{ln}] No data, skipping'); continue
    tr_in, tr_tg, tr_sc = ctrl_data[ln]['train']
    va_in, va_tg, va_sc = ctrl_data[ln]['val']

    tr_ds = TensorDataset(torch.tensor(tr_in), torch.tensor(tr_tg), torch.tensor(tr_sc, dtype=torch.long))
    va_ds = TensorDataset(torch.tensor(va_in), torch.tensor(va_tg), torch.tensor(va_sc, dtype=torch.long))
    tr_dl = DataLoader(tr_ds, CTRL_BATCH, shuffle=True,  pin_memory=True)
    va_dl = DataLoader(va_ds, CTRL_BATCH*2, pin_memory=True)

    model.to(DEVICE)
    opt  = torch.optim.Adam(model.parameters(), lr=CTRL_LR)
    sch  = torch.optim.lr_scheduler.ReduceLROnPlateau(opt, patience=5, factor=0.5)
    crit = nn.MSELoss()
    best_val, pat, best_st = float('inf'), 0, None
    tr_l, va_l = [], []

    print(f'\n[{ln}] train={len(tr_ds):,}  val={len(va_ds):,}  in_dim={tr_in.shape[-1]}')
    for ep in range(CTRL_EPOCHS):
        model.train(); tl = 0.0
        for x, y, sc in tr_dl:
            x, y, sc = x.to(DEVICE), y.to(DEVICE), sc.to(DEVICE)
            pred, _  = model(x, sc)
            loss     = crit(pred, y)
            opt.zero_grad(); loss.backward()
            nn.utils.clip_grad_norm_(model.parameters(), 1.0); opt.step()
            tl += loss.item() * len(x)
        tl /= len(tr_ds)
        model.eval(); vl = 0.0
        with torch.no_grad():
            for x, y, sc in va_dl:
                x, y, sc = x.to(DEVICE), y.to(DEVICE), sc.to(DEVICE)
                vl += crit(model(x,sc)[0], y).item() * len(x)
        vl /= len(va_ds)
        sch.step(vl); tr_l.append(tl); va_l.append(vl)
        if vl < best_val:
            best_val, pat = vl, 0
            best_st = {k: v.cpu().clone() for k,v in model.state_dict().items()}
        else:
            pat += 1
        if ep % 20 == 0 or pat >= CTRL_PATIENCE:
            print(f'  ep{ep:3d}: train={tl:.5f}  val={vl:.5f}  pat={pat}')
        if pat >= CTRL_PATIENCE:
            print(f'  Early stop at ep {ep}'); break

    assert best_st is not None
    model.load_state_dict(best_st); model.eval()
    torch.save({'model_state': model.state_dict(), 'loop': ln, 'best_val': best_val},
               OUTPUT_DIR/'models'/f'ctrl_{ln}.pt')
    np.save(OUTPUT_DIR/f'ctrl_{ln}_train_losses.npy', np.array(tr_l))
    np.save(OUTPUT_DIR/f'ctrl_{ln}_val_losses.npy',   np.array(va_l))
    ctrl_losses[ln] = (tr_l, va_l)
    print(f'  [{ln}] best val={best_val:.5f}')

In [ ]:
# Plot: Controller loss curves
n = len(ctrl_losses)
if n > 0:
    fig, axes = plt.subplots(2, n, figsize=(4*n, 7), squeeze=False)
    for col, (ln, (trl, val)) in enumerate(ctrl_losses.items()):
        ep = np.arange(1, len(trl)+1)
        axes[0][col].plot(ep, trl, color='steelblue', lw=1.1, label='Train')
        axes[0][col].plot(ep, val, color='orange',    lw=1.1, label='Val')
        axes[0][col].set_title(f'{ln} Controller'); axes[0][col].legend(fontsize=7)
        axes[0][col].set_ylabel('MSE Loss' if col==0 else ''); axes[0][col].grid(True,alpha=0.2)
        axes[1][col].semilogy(ep, trl, color='steelblue', lw=1.1)
        axes[1][col].semilogy(ep, val, color='orange',    lw=1.1)
        axes[1][col].set_xlabel('Epoch')
        axes[1][col].set_ylabel('Loss (log)' if col==0 else ''); axes[1][col].grid(True,alpha=0.2)
    fig.suptitle('Controller Loss Curves', fontsize=12)
    fig.tight_layout(); plt.show()
    fig.savefig(OUTPUT_DIR/'plots'/'ctrl_loss_curves.png', dpi=150, bbox_inches='tight')

In [ ]:
# Plot: Controller predictions vs actual (validation set, first window per loop)
loops_to_plot = [ln for ln in LOOPS_DEF if ln in ctrl_models and ln in ctrl_data]
n = len(loops_to_plot)
fig, axes = plt.subplots(n, 1, figsize=(14, 3*n), sharex=False, squeeze=False)

for row, ln in enumerate(loops_to_plot):
    model = ctrl_models[ln].to(DEVICE).eval()
    va_in, va_tg, va_sc = ctrl_data[ln]['val']
    idx = 0
    x   = torch.tensor(va_in[[idx]], dtype=torch.float32).to(DEVICE)
    sc  = torch.tensor(va_sc[[idx]], dtype=torch.long).to(DEVICE)
    with torch.no_grad():
        pred, _ = model(x, sc)
    pred_np = pred[0].cpu().numpy()
    true_np = va_tg[idx]
    t = np.arange(len(pred_np))
    ax = axes[row][0]
    ax.plot(t, true_np, color='green', lw=1.1, label='Actual CV')
    ax.plot(t, pred_np, color='crimson', lw=1.1, linestyle='--', label='Predicted CV')
    ax.fill_between(t, true_np, pred_np, alpha=0.15, color='red')
    rmse = float(np.sqrt(np.mean((true_np - pred_np)**2)))
    ax.set_title(f'{ln} — Val window  RMSE={rmse:.5f}', fontsize=9)
    ax.set_ylabel('Scaled CV'); ax.legend(fontsize=7); ax.grid(True, alpha=0.2)

axes[-1][0].set_xlabel('Timestep')
fig.suptitle('Controller Predictions vs Actual (Validation)', fontsize=12)
fig.tight_layout(); plt.show()
fig.savefig(OUTPUT_DIR/'plots'/'ctrl_predictions.png', dpi=120, bbox_inches='tight')

---
## Step 6 — Phase B: Train Plant LSTM

Single MIMO model: `[CVs + aux, PV_current]` + scenario embedding → next 5 PVs  
Training uses **scheduled sampling**: pure teacher-forcing for epochs 0-9, then ramps to 50% autoregressive by epoch 100.

In [ ]:
PLANT_LR       = 1e-3
PLANT_EPOCHS   = 150
PLANT_BATCH    = 256
PLANT_PATIENCE = 20
LAMBDA_CAUSAL  = 0.1
SS_START, SS_END, SS_MAX = 10, 100, 0.5

def get_ss_ratio(epoch):
    if epoch < SS_START: return 0.0
    if epoch >= SS_END:  return SS_MAX
    return ((epoch - SS_START) / (SS_END - SS_START)) * SS_MAX

plant_model = PlantLSTM(N_PLANT_IN, N_PLANT_OUT, hidden_dim=256,
                         n_scenarios=N_SCENARIOS, emb_dim=EMB_DIM*2).to(DEVICE)
causal_fn   = CausalLoss(lambda_causal=LAMBDA_CAUSAL).to(DEVICE)

tr_ds = TensorDataset(torch.tensor(cv_tr), torch.tensor(pi_tr),
                       torch.tensor(pte_tr), torch.tensor(ptr_tr),
                       torch.tensor(sc_tr, dtype=torch.long))
va_ds = TensorDataset(torch.tensor(cv_va), torch.tensor(pi_va),
                       torch.tensor(pte_va), torch.tensor(ptr_va),
                       torch.tensor(sc_va, dtype=torch.long))
tr_dl = DataLoader(tr_ds, PLANT_BATCH, shuffle=True,  pin_memory=True)
va_dl = DataLoader(va_ds, PLANT_BATCH*2, pin_memory=True)

opt = torch.optim.Adam(plant_model.parameters(), lr=PLANT_LR)
sch = torch.optim.lr_scheduler.CosineAnnealingLR(opt, PLANT_EPOCHS)

best_val, pat, best_st = float('inf'), 0, None
train_losses, val_losses, ss_ratios, mse_losses, causal_losses = [], [], [], [], []

print(f'Plant: n_in={N_PLANT_IN}  n_out={N_PLANT_OUT}  train={len(tr_ds):,}  val={len(va_ds):,}')
for ep in range(PLANT_EPOCHS):
    ss  = get_ss_ratio(ep); t0 = time.time()
    plant_model.train(); tl = mse_l = caus_l = 0.0
    for cv, pi, pte, ptr, sc in tr_dl:
        cv,pi,pte,ptr,sc = cv.to(DEVICE),pi.to(DEVICE),pte.to(DEVICE),ptr.to(DEVICE),sc.to(DEVICE)
        pv_pred   = plant_model(cv, pi, sc, teacher_pv=pte, ss_ratio=ss)
        tot,mse,c = combined_loss(pv_pred, ptr, cv, causal_fn)
        opt.zero_grad(); tot.backward()
        nn.utils.clip_grad_norm_(plant_model.parameters(), 1.0); opt.step()
        tl += tot.item()*len(cv); mse_l += mse.item()*len(cv); caus_l += c.item()*len(cv)
    tl /= len(tr_ds); mse_l /= len(tr_ds); caus_l /= len(tr_ds)
    plant_model.eval(); vl = 0.0
    with torch.no_grad():
        for cv,pi,pte,ptr,sc in va_dl:
            cv,pi,ptr,sc = cv.to(DEVICE),pi.to(DEVICE),ptr.to(DEVICE),sc.to(DEVICE)
            vl += nn.functional.mse_loss(plant_model(cv,pi,sc,ss_ratio=1.0),ptr).item()*len(cv)
    vl /= len(va_ds); sch.step()
    train_losses.append(tl); val_losses.append(vl); ss_ratios.append(ss)
    mse_losses.append(mse_l); causal_losses.append(caus_l)
    if vl < best_val:
        best_val, pat = vl, 0
        best_st = {k: v.cpu().clone() for k,v in plant_model.state_dict().items()}
    else: pat += 1
    if ep % 10 == 0 or pat >= PLANT_PATIENCE:
        print(f'  ep{ep:3d}: train={tl:.5f}  val={vl:.5f}  ss={ss:.2f}  pat={pat}  ({time.time()-t0:.1f}s)')
    if pat >= PLANT_PATIENCE:
        print(f'  Early stop at ep {ep}'); break

assert best_st is not None
plant_model.load_state_dict(best_st); plant_model.eval()
torch.save({'model_state': plant_model.state_dict(), 'n_in': N_PLANT_IN, 'n_out': N_PLANT_OUT,
            'best_val': best_val}, OUTPUT_DIR/'models'/'plant.pt')
np.save(OUTPUT_DIR/'plant_train_losses.npy', np.array(train_losses))
np.save(OUTPUT_DIR/'plant_val_losses.npy',   np.array(val_losses))
print(f'\nPlant trained. Best val={best_val:.5f}')

In [ ]:
ep = np.arange(1, len(train_losses)+1)
fig, axes = plt.subplots(1, 3, figsize=(16, 4))

axes[0].plot(ep, train_losses, color='steelblue', lw=1.2, label='Train (total)')
axes[0].plot(ep, val_losses,   color='orange',    lw=1.2, label='Val (autoregressive)')
axes[0].set_xlabel('Epoch'); axes[0].set_ylabel('Loss')
axes[0].set_title('Plant LSTM — Loss Curves'); axes[0].legend(); axes[0].grid(True, alpha=0.2)

axes[1].semilogy(ep, train_losses, color='steelblue', lw=1.2)
axes[1].semilogy(ep, val_losses,   color='orange',    lw=1.2)
axes[1].set_xlabel('Epoch'); axes[1].set_ylabel('Loss (log)')
axes[1].set_title('Plant LSTM — Log Scale'); axes[1].grid(True, alpha=0.2)

ax2 = axes[2].twinx()
axes[2].plot(ep, mse_losses,    color='steelblue', lw=1.1, label='MSE')
axes[2].plot(ep, causal_losses, color='crimson',   lw=1.1, label='Causal penalty')
ax2.plot(ep, ss_ratios, color='gray', lw=1.1, linestyle=':', label='SS ratio')
axes[2].set_xlabel('Epoch'); axes[2].set_ylabel('Loss component')
ax2.set_ylabel('SS ratio', color='gray')
axes[2].set_title('MSE vs Causal Loss + SS Ratio')
lines1, labs1 = axes[2].get_legend_handles_labels()
lines2, labs2 = ax2.get_legend_handles_labels()
axes[2].legend(lines1+lines2, labs1+labs2, fontsize=8); axes[2].grid(True, alpha=0.2)

fig.tight_layout(); plt.show()
fig.savefig(OUTPUT_DIR/'plots'/'plant_training.png', dpi=150, bbox_inches='tight')
print('Saved plant_training.png')

---
## Step 7 — Closed-Loop Rollout

Run the full controller + plant loop autoregressively:
1. **Warmup (300 steps)** on real data to initialise LSTM hidden states
2. At each step: controllers predict CVs → plant predicts next PVs → feed back

In [ ]:
WARMUP = 300

def _warmup(df_w, sc_id):
    sc_t = torch.tensor([sc_id], dtype=torch.long, device=DEVICE)
    ctrl_h = {}
    for ln, model in ctrl_models.items():
        d    = LOOPS_DEF[ln]
        cols = [d['sp'], d['pv']] + ([d['cv_fb']] if d['cv_fb'] else []) + [d['cv']]
        cols = [c for c in cols if c in SENSOR_COLS]
        if not cols: continue
        cidx   = [SENSOR_COLS.index(c) for c in cols]
        raw    = df_w[SENSOR_COLS].values.astype(np.float32)
        scaled = scaler.transform(raw)[:, cidx]
        x_w    = torch.tensor(scaled[:-1, :-1], dtype=torch.float32).unsqueeze(0).to(DEVICE)
        with torch.no_grad(): _, h = model(x_w, sc_t)
        ctrl_h[ln] = h
    # Plant warmup
    in_c   = [c for c in PLANT_IN_COLS  if c in SENSOR_COLS]
    out_c  = [c for c in PLANT_OUT_COLS if c in SENSOR_COLS]
    pcols  = in_c + out_c
    pidx   = [SENSOR_COLS.index(c) for c in pcols]
    raw    = df_w[SENSOR_COLS].values.astype(np.float32)
    scaled = scaler.transform(raw)[:, pidx]
    ni     = len(in_c)
    plant_h = None
    with torch.no_grad():
        for t in range(len(df_w)):
            cv_t = torch.tensor(scaled[t,:ni], dtype=torch.float32).unsqueeze(0).to(DEVICE)
            pv_t = torch.tensor(scaled[t,ni:], dtype=torch.float32).unsqueeze(0).to(DEVICE)
            _, plant_h = plant_model.step(cv_t, pv_t, sc_t, plant_h)
    pv_cur = torch.tensor(scaled[-1, ni:], dtype=torch.float32).unsqueeze(0).to(DEVICE)
    return ctrl_h, plant_h, pv_cur

def rollout_fn(df, start, horizon, sc_id):
    if start + WARMUP + horizon >= len(df): return None
    sc_t   = torch.tensor([sc_id], dtype=torch.long, device=DEVICE)
    ctrl_h, plant_h, pv_cur = _warmup(df.iloc[start:start+WARMUP], sc_id)
    in_c   = [c for c in PLANT_IN_COLS  if c in SENSOR_COLS]
    out_c  = [c for c in PLANT_OUT_COLS if c in SENSOR_COLS]
    pcols  = in_c + out_c; ni = len(in_c)
    pidx   = [SENSOR_COLS.index(c) for c in pcols]
    cv_idx = {ln: SENSOR_COLS.index(LOOPS_DEF[ln]['cv'])
               for ln in LOOPS_DEF if LOOPS_DEF[ln]['cv'] in SENSOR_COLS}
    pred_pvs, actual_pvs, pred_cvs = [], [], []
    for t in range(horizon):
        abs_t = start + WARMUP + t
        row_s = scaler.transform(df.iloc[[abs_t]][SENSOR_COLS].values.astype(np.float32))[0]
        cvs   = {}
        for ln, model in ctrl_models.items():
            d = LOOPS_DEF[ln]
            if d['sp'] not in SENSOR_COLS: continue
            sp_val = torch.tensor([[row_s[SENSOR_COLS.index(d['sp'])]]], dtype=torch.float32).to(DEVICE)
            pv_idx_l = [c['pv'] for c in [LOOPS_DEF[ln]]]
            pv_idx_i = PLANT_OUT_COLS.index(d['pv']) if d['pv'] in PLANT_OUT_COLS else None
            pv_val   = pv_cur[:, pv_idx_i:pv_idx_i+1] if pv_idx_i is not None else torch.zeros(1,1,device=DEVICE)
            if d['cv_fb'] and d['cv_fb'] in SENSOR_COLS:
                fb = torch.tensor([[row_s[SENSOR_COLS.index(d['cv_fb'])]]], dtype=torch.float32).to(DEVICE)
                x_ctrl = torch.cat([sp_val, pv_val, fb], dim=-1)
            else:
                x_ctrl = torch.cat([sp_val, pv_val], dim=-1)
            with torch.no_grad():
                cv_pred, ctrl_h[ln] = model.step(x_ctrl, sc_t, ctrl_h.get(ln))
            cvs[ln] = cv_pred.item()
        row_s2 = row_s.copy()
        for ln, cv_val in cvs.items():
            if ln in cv_idx: row_s2[cv_idx[ln]] = cv_val
        cv_aux = torch.tensor(row_s2[[SENSOR_COLS.index(c) for c in in_c if c in SENSOR_COLS]],
                               dtype=torch.float32).unsqueeze(0).to(DEVICE)
        with torch.no_grad():
            pv_pred, plant_h = plant_model.step(cv_aux, pv_cur, sc_t, plant_h)
        pv_cur = pv_pred.detach()
        # Inverse transform
        full = np.zeros((1, len(SENSOR_COLS)), np.float32)
        pv_idx_all = [SENSOR_COLS.index(c) for c in out_c]
        for i, ci in enumerate(pv_idx_all): full[0, ci] = pv_pred[0,i].item()
        pv_raw  = scaler.inverse_transform(full)[0, pv_idx_all]
        act_row = df.iloc[abs_t+1][SENSOR_COLS].values.astype(np.float32)
        act_raw = scaler.inverse_transform(act_row.reshape(1,-1))[0, pv_idx_all]
        pred_pvs.append(pv_raw); actual_pvs.append(act_raw)
        pred_cvs.append([cvs.get(ln,0.0) for ln in LOOPS_DEF])
    return np.array(pred_pvs), np.array(actual_pvs), np.array(pred_cvs), out_c

print('Rollout functions defined.')

In [ ]:
# Run rollout for normal scenario (sc=0)
df_test  = files.get('train4', list(files.values())[0])
HORIZON  = 1800
SC_ID    = 0

print(f'Running rollout: scenario={SCENARIO_NAMES[SC_ID]}  horizon={HORIZON}s...')
result = rollout_fn(df_test, start=0, horizon=HORIZON, sc_id=SC_ID)
if result is None:
    print('Not enough data for warmup + horizon. Reduce horizon or use a longer CSV.')
else:
    pred_pvs, actual_pvs, pred_cvs, pv_cols_out = result
    rmse = np.sqrt(((pred_pvs - actual_pvs)**2).mean(axis=0))
    print('\nRMSE per PV:')
    for col, r in zip(pv_cols_out, rmse):
        print(f'  {col}: {r:.4f}')
    # Save
    np.savez_compressed(OUTPUT_DIR/f'generated_sc{SC_ID}_h{HORIZON}.npz',
                        pred_pvs=pred_pvs, actual_pvs=actual_pvs, pred_cvs=pred_cvs,
                        pv_cols=pv_cols_out, scenario=SC_ID)
    print(f'Saved generated_sc{SC_ID}_h{HORIZON}.npz')

In [ ]:
if result is not None:
    pred_pvs, actual_pvs, pred_cvs, pv_cols_out = result
    n_pv = len(pv_cols_out)
    t    = np.arange(len(pred_pvs))
    fig, axes = plt.subplots(n_pv, 1, figsize=(16, 3*n_pv), sharex=True, squeeze=False)
    for i, pv in enumerate(pv_cols_out):
        ax = axes[i][0]
        ax.plot(t, actual_pvs[:,i], color='green', lw=1.1, label='Actual')
        ax.plot(t, pred_pvs[:,i],   color='crimson', lw=1.1, linestyle='--', label='Predicted')
        ax.fill_between(t, actual_pvs[:,i], pred_pvs[:,i], alpha=0.12, color='red')
        rmse_i = float(np.sqrt(np.mean((actual_pvs[:,i]-pred_pvs[:,i])**2)))
        ax.set_title(f'{pv}  RMSE={rmse_i:.4f}', fontsize=9)
        ax.set_ylabel('Value'); ax.legend(fontsize=7); ax.grid(True, alpha=0.2)
    axes[0][0].legend(fontsize=8)
    axes[-1][0].set_xlabel('Time (s)')
    fig.suptitle(f'PV Predicted vs Actual — {SCENARIO_NAMES[SC_ID]} (horizon={HORIZON}s)', fontsize=12)
    fig.tight_layout(); plt.show()
    fig.savefig(OUTPUT_DIR/'plots'/'rollout_pv_timeseries.png', dpi=120, bbox_inches='tight')

In [ ]:
if result is not None:
    loop_names = list(LOOPS_DEF.keys())
    t = np.arange(len(pred_cvs))
    fig, axes = plt.subplots(len(loop_names), 1, figsize=(16, 2.5*len(loop_names)), sharex=True, squeeze=False)
    for i, ln in enumerate(loop_names):
        ax = axes[i][0]
        ax.plot(t, pred_cvs[:,i], color=SCENARIO_COLORS[SC_ID], lw=1.0)
        ax.set_ylabel(f'{ln} CV'); ax.grid(True, alpha=0.2)
    axes[-1][0].set_xlabel('Time (s)')
    fig.suptitle(f'Controller CV Outputs — {SCENARIO_NAMES[SC_ID]}', fontsize=12)
    fig.tight_layout(); plt.show()
    fig.savefig(OUTPUT_DIR/'plots'/'rollout_cv_outputs.png', dpi=120, bbox_inches='tight')

---
## Step 8 — Level 1 & 2 Validation

**Level 1**: Error distribution histograms + Q-Q plots per PV  
**Level 2**: KS test comparing SP-tracking error between actual and predicted

In [ ]:
# Multi-horizon NRMSE validation
HORIZONS = [300, 600, 900, 1800]
NRMSE_THRESHOLD = 0.10

def nrmse(y_true, y_pred):
    r = float(y_true.max() - y_true.min())
    return float(np.sqrt(np.mean((y_true-y_pred)**2)) / r) if r > 1e-10 else 0.0

gate_results = {}
print(f'{"Signal":<14} ' + '  '.join(f'{h:>8}s' for h in HORIZONS))
for H in HORIZONS:
    res = rollout_fn(df_test, start=0, horizon=H, sc_id=0)
    if res is None: continue
    pp, ap, _, out_c = res
    gate_results[H] = {pv: nrmse(ap[:,i], pp[:,i]) for i,pv in enumerate(out_c)}

for pv in (pv_cols_out if result else PV_COLS):
    row = f'  {pv:<14}'
    for H in HORIZONS:
        v = gate_results.get(H,{}).get(pv)
        row += f'  {"OK" if v and v<NRMSE_THRESHOLD else "!!"}  {v:.4f}' if v else '  N/A  '
    print(row)

# Bar chart
if gate_results:
    avail_h = sorted(gate_results)
    pvs_out = list(next(iter(gate_results.values())).keys())
    x = np.arange(len(pvs_out)); width = 0.8/len(avail_h)
    cmap = plt.cm.get_cmap('Blues', len(avail_h)+2)
    fig, ax = plt.subplots(figsize=(10, 5))
    for i, H in enumerate(avail_h):
        vals   = [gate_results[H].get(pv,0) for pv in pvs_out]
        offset = (i - len(avail_h)/2 + 0.5) * width
        ax.bar(x+offset, vals, width*0.9, color=cmap(i+2), label=f'{H}s', edgecolor='white')
    ax.axhline(NRMSE_THRESHOLD, color='red', linestyle='--', lw=1.2, label=f'Threshold ({NRMSE_THRESHOLD})')
    ax.set_xticks(x); ax.set_xticklabels([p[:14] for p in pvs_out], fontsize=9)
    ax.set_ylabel('NRMSE'); ax.set_title('Multi-Horizon NRMSE — Normal Scenario')
    ax.legend(fontsize=8); ax.grid(True, alpha=0.2, axis='y')
    fig.tight_layout(); plt.show()
    fig.savefig(OUTPUT_DIR/'plots'/'multi_horizon_nrmse.png', dpi=150, bbox_inches='tight')

In [ ]:
# Level 1: Error distributions + Q-Q
if result is not None:
    pred_pvs, actual_pvs, _, pv_cols_out = result
    n_pv = len(pv_cols_out)
    fig, axes = plt.subplots(2, n_pv, figsize=(4*n_pv, 8), squeeze=False)
    fig.suptitle('Level 1 — PV Error Distributions (Normal scenario)', fontsize=12)
    for i, pv in enumerate(pv_cols_out):
        ln  = next((k for k,v in LOOPS_DEF.items() if v['pv']==pv), pv)
        err = actual_pvs[:,i] - pred_pvs[:,i]
        # Histogram
        ax = axes[0][i]
        ax.hist(err, bins=80, density=True, alpha=0.75, color='steelblue', edgecolor='none')
        ax.axvline(0, color='red', linestyle='--', lw=1.2)
        ax.axvline(err.mean(), color='black', linestyle=':', lw=1.0,
                   label=f'mean={err.mean():.3f}')
        ax.set_title(ln, fontsize=9); ax.set_xlabel('Error')
        ax.set_ylabel('Density' if i==0 else ''); ax.legend(fontsize=7); ax.grid(True, alpha=0.2)
        # Q-Q
        ax = axes[1][i]
        probplot(err, dist='norm', plot=ax)
        ax.set_title(f'{ln} Q-Q', fontsize=9); ax.grid(True, alpha=0.2)
    fig.tight_layout(); plt.show()
    fig.savefig(OUTPUT_DIR/'plots'/'level1_error_distributions.png', dpi=150, bbox_inches='tight')
    print('Saved level1_error_distributions.png')

In [ ]:
# Level 2: KS test on SP tracking error
if result is not None:
    pred_pvs, actual_pvs, _, pv_cols_out = result
    T = len(pred_pvs)
    print('Level 2 — KS Test on SP Tracking Error')
    print(f'  {"Loop":<6} {"KS":>8}  {"p":>10}  Result')
    for ln, d in LOOPS_DEF.items():
        pv = d['pv']; sp = d['sp']
        if pv not in pv_cols_out or sp not in df_test.columns: continue
        pi       = pv_cols_out.index(pv)
        sp_vals  = df_test[sp].iloc[:T].values[:T].astype(np.float32)
        if len(sp_vals) < T: continue
        real_err  = sp_vals - actual_pvs[:, pi]
        synth_err = sp_vals - pred_pvs[:, pi]
        n    = min(5000, T)
        res2 = ks_2samp(real_err[:n], synth_err[:n])  # type: ignore
        ks: float = float(res2[0])  # type: ignore
        p:  float = float(res2[1])  # type: ignore
        flag = 'PASS' if ks < 0.1 else 'FAIL'
        print(f'  {ln:<6} {ks:>8.4f}  {p:>10.4e}  {flag}')

---
## Step 9 — Full Evaluation Plots

RMSE per PV, error boxplot, scatter true vs predicted, residual Q-Q.

In [ ]:
# Run rollouts for all scenarios and collect results
rollouts = {}
for sc_id in range(5):
    r = rollout_fn(df_test, start=0, horizon=HORIZON, sc_id=sc_id)
    if r is not None:
        pred_p, actual_p, pred_c, out_c = r
        rollouts[sc_id] = {'pred_pvs': pred_p, 'actual_pvs': actual_p,
                           'pred_cvs': pred_c, 'pv_cols': out_c}
        print(f'Scenario {sc_id} ({SCENARIO_NAMES[sc_id]}): rollout OK')
    else:
        print(f'Scenario {sc_id}: not enough data')

In [ ]:
# RMSE per PV per scenario — bar chart
if rollouts:
    scs   = sorted(rollouts)
    pvs   = rollouts[scs[0]]['pv_cols']
    n_pv  = len(pvs)
    x     = np.arange(n_pv)
    width = 0.8 / len(scs)
    fig, ax = plt.subplots(figsize=(max(8, 2*n_pv), 5))
    for i, sc in enumerate(scs):
        pred   = rollouts[sc]['pred_pvs']
        actual = rollouts[sc]['actual_pvs']
        rmse   = np.sqrt(((pred - actual)**2).mean(axis=0))
        offset = (i - len(scs)/2 + 0.5) * width
        bars   = ax.bar(x+offset, rmse, width*0.9, color=SCENARIO_COLORS[sc],
                        label=SCENARIO_NAMES[sc], edgecolor='white', alpha=0.85)
        for bar, r in zip(bars, rmse):
            ax.text(bar.get_x()+bar.get_width()/2, bar.get_height()+0.001,
                    f'{r:.3f}', ha='center', va='bottom', fontsize=7)
    ax.set_xticks(x); ax.set_xticklabels([p[:14] for p in pvs], fontsize=9)
    ax.set_ylabel('RMSE (raw units)'); ax.set_title('RMSE per PV x Scenario')
    ax.legend(fontsize=9); ax.grid(True, alpha=0.2, axis='y')
    fig.tight_layout(); plt.show()
    fig.savefig(OUTPUT_DIR/'plots'/'rmse_per_pv_scenario.png', dpi=150, bbox_inches='tight')

In [ ]:
# Squared error boxplot per PV per scenario
if rollouts:
    scs  = sorted(rollouts); pvs = rollouts[scs[0]]['pv_cols']
    fig, axes = plt.subplots(1, len(pvs), figsize=(4*len(pvs), 5), sharey=False)
    axes = [axes] if len(pvs)==1 else list(axes)
    for ax, (i, pv) in zip(axes, enumerate(pvs)):
        data, labels, colors = [], [], []
        for sc in scs:
            err = (rollouts[sc]['pred_pvs'][:,i] - rollouts[sc]['actual_pvs'][:,i])**2
            data.append(err); labels.append(SCENARIO_NAMES[sc]); colors.append(SCENARIO_COLORS[sc])
        bp = ax.boxplot(data, patch_artist=True, showfliers=False, medianprops=dict(color='black',lw=1.5))
        for patch, col in zip(bp['boxes'], colors):
            patch.set_facecolor(col); patch.set_alpha(0.7)
        ax.set_title(pv[:16], fontsize=8)
        ax.set_xticklabels(labels, fontsize=7, rotation=25)
        ax.set_ylabel('Squared error' if i==0 else ''); ax.grid(True, alpha=0.2, axis='y')
    fig.suptitle('Reconstruction Error per PV x Scenario', fontsize=11)
    fig.tight_layout(); plt.show()
    fig.savefig(OUTPUT_DIR/'plots'/'error_boxplot.png', dpi=150, bbox_inches='tight')

In [ ]:
# Scatter true vs predicted (normal scenario)
if 0 in rollouts:
    pred   = rollouts[0]['pred_pvs']
    actual = rollouts[0]['actual_pvs']
    pvs    = rollouts[0]['pv_cols']
    fig, axes = plt.subplots(1, len(pvs), figsize=(5*len(pvs), 5), squeeze=False)
    for col, pv in enumerate(pvs):
        ax = axes[0][col]
        tv = actual[:,col]; pv_v = pred[:,col]
        rmse = float(np.sqrt(np.mean((tv-pv_v)**2)))
        ss_r = float(np.sum((tv-pv_v)**2)); ss_t = float(np.sum((tv-tv.mean())**2))
        r2   = 1.0 - ss_r/ss_t if ss_t > 0 else float('nan')
        lo   = min(float(tv.min()), float(pv_v.min()))
        hi   = max(float(tv.max()), float(pv_v.max()))
        ax.scatter(tv, pv_v, s=2, alpha=0.3, color='steelblue', rasterized=True)
        ax.plot([lo,hi],[lo,hi], color='red', lw=1.5, label='y=x')
        ax.set_xlabel('Actual'); ax.set_ylabel('Predicted' if col==0 else '')
        ax.set_title(pv[:16], fontsize=9)
        ax.text(0.05, 0.93, f'R2={r2:.4f}\nRMSE={rmse:.4f}',
                transform=ax.transAxes, fontsize=8, va='top',
                bbox=dict(boxstyle='round', facecolor='white', alpha=0.85))
        ax.grid(True, alpha=0.2); ax.set_aspect('equal', adjustable='box')
    fig.suptitle('True vs Predicted — Normal Scenario', fontsize=12)
    fig.tight_layout(); plt.show()
    fig.savefig(OUTPUT_DIR/'plots'/'scatter_true_vs_pred.png', dpi=150, bbox_inches='tight')

In [ ]:
# Residual Q-Q per PV (normal scenario)
if 0 in rollouts:
    pred   = rollouts[0]['pred_pvs']
    actual = rollouts[0]['actual_pvs']
    pvs    = rollouts[0]['pv_cols']
    fig, axes = plt.subplots(1, len(pvs), figsize=(5*len(pvs), 5), squeeze=False)
    for col, pv in enumerate(pvs):
        resid = actual[:,col] - pred[:,col]
        resid -= resid.mean()
        ax = axes[0][col]
        probplot(resid, dist='norm', plot=ax)
        ax.set_title(pv[:16], fontsize=9); ax.grid(True, alpha=0.2)
    fig.suptitle('Residual Q-Q Plots — Normal Scenario', fontsize=12)
    fig.tight_layout(); plt.show()
    fig.savefig(OUTPUT_DIR/'plots'/'residual_qq.png', dpi=150, bbox_inches='tight')
    print('All evaluation plots saved to', OUTPUT_DIR/'plots')